# ZMART target acquisition

Run the cells from top to bottom. Edit the setup values, select the requested LAS X jobs when asked, and inspect the plots before acquiring targets.

## 1 · Setup and connect
Edit the analysis repo, then run.

In [ ]:
import sys
sys.path.insert(0, "workflows/target_acquisition")
from _bootstrap import Path, workflow

ANALYSIS_REPO = Path("C:/code/smart-analysis")
SIMULATE_IMAGES = False

zmart_controller = workflow.connect("leica")
ROOT = Path(zmart_controller.run_procedure({"name": "get_root"})["root"])
ROOT

## 2 · Set origin
Move to the run origin first. This makes the current position `(0, 0, 0)`.

In [ ]:
zmart_controller.set_origin()

## 3a · Overview job

Select the low-magnification overview job in LAS X, then run.

In [ ]:
overview_state = zmart_controller.get_state()
overview_state["observed"]

## 3b · Target job

Select the high-magnification target job in LAS X, then run.

In [ ]:
target_state = zmart_controller.get_state()
target_state["observed"]

## 4 · Initial positions
Read overview tile centres from the controller.

In [ ]:
positions = zmart_controller.run_procedure({"name": "get_positions"})["positions"]
print(len(positions), "overview positions")

## 5 · Focus

Pick focus points on the map, then press **Measure focus**. The microscope visits each point, runs the autofocus job there, and the fitted focus map appears as a heatmap in the same figure.

- **Left-click** adds a focus point; **right-click** removes the nearest one.
- Points already placed in LAS X are pre-filled.
- Clicking needs an interactive matplotlib backend: in JupyterLab, run `%matplotlib widget` once in an empty cell first (this needs the `ipympl` package). Without it the figure is static — add points in code with `picker.add_point(x, y)` and call `picker.measure()`.

In [ ]:
picker = workflow.pick_focus_points(zmart_controller, positions)

In [ ]:
focus = picker.require_focus()
workflow.plot_focus_surface(focus, save_path=ROOT / "focus_surface.png", show=True)
print("focus model:", focus.model)

## 6 · Overview

Capture one overview at each position, then browse them as one map. Every tile sits at its real stage position; pan and zoom with the toolbar. The controls on the right work per channel: pick a channel, toggle **show**, step its **colour**, and drag the **display range** slider for brightness and contrast (the same min/max control as Fiji's B&C).

In [ ]:
overview_records = workflow.run_overview(zmart_controller, positions, state=overview_state, focus=focus)
print(len(overview_records), "overviews captured")
if SIMULATE_IMAGES:
    n = workflow.hijack_if_simulating(
        overview_records,
        simulate=True,
    )
    print(n, "overview images replaced")

In [ ]:
overviews = workflow.overview_inputs_from_records(positions, overview_records, focus=focus)
viewer = workflow.view_overview(overviews)

## 7 · Discover targets

Segment the overview images, then decide which cells become targets in the explorer:

- the radio lists put any measured feature on the plot's x and y axes (position, area, brightness, ...);
- gate with the two range sliders (thresholds on the current axes) and/or by drawing a lasso around points with the mouse — gated points stay blue, the rest fade out;
- hover near a point to see that cell's image crop in the side panel.

`explorer.gated` is the list the next step samples from.

In [ ]:
engine = workflow.load_analysis_engine(ANALYSIS_REPO)
targets = workflow.discover_targets(engine, overviews)
print(len(targets), "targets discovered")

In [ ]:
explorer = workflow.explore_targets(targets, overviews)

## 8 · Acquire targets

Type how many targets to acquire and press **Acquire**. That many cells are drawn at random from the gate, re-imaged at the target job, and shown below as pairs — the overview crop (left) and the new high-magnification image (right), both over the same physical window so the cell appears at the same scale in both.

In [ ]:
gallery = workflow.acquire_gallery(
    zmart_controller,
    explorer,
    overviews,
    state=target_state,
    focus=focus,
    after_acquire=lambda records: workflow.hijack_if_simulating(records, simulate=SIMULATE_IMAGES),
)

## 9 · Summary
Write the summary and run map.

In [ ]:
summary = workflow.write_run_report(
    ROOT,
    positions=positions,
    focus=focus,
    overview_records=overview_records,
    targets=gallery.picked,
)
summary

In [ ]:
zmart_controller.disconnect()